In [5]:
import pandas as pd
import os
import unicodedata
import re

In [1]:
RAW_DATA_PATH = "data/raw"
PROCESSED_DATA_PATH = "data/processed"
OUTPUT_FILE = "cleaned_parallel_data.csv"

In [2]:
def normalize_text(text):
    text = str(text).strip()
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text)
    return text

In [6]:
def load_and_clean(file_path, src_col, tgt_col):
    df = pd.read_csv(file_path)

    df = df[[src_col, tgt_col]]
    df.dropna(inplace=True)

    df[src_col] = df[src_col].apply(normalize_text)
    df[tgt_col] = df[tgt_col].apply(normalize_text)

    # Length filtering (avoid garbage lines)
    df = df[
        (df[src_col].str.len() > 2) &
        (df[tgt_col].str.len() > 2) &
        (df[src_col].str.len() < 300) &
        (df[tgt_col].str.len() < 300)
    ]

    return df

In [7]:
def main():
    os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

    en_hi = load_and_clean(
        f"{RAW_DATA_PATH}/en_hi.csv",
        "english",
        "hindi"
    )

    hi_en = load_and_clean(
        f"{RAW_DATA_PATH}/hi_en.csv",
        "hindi",
        "english"
    )

    combined = pd.concat([en_hi, hi_en], ignore_index=True)
    combined.drop_duplicates(inplace=True)

    output_path = f"{PROCESSED_DATA_PATH}/{OUTPUT_FILE}"
    combined.to_csv(output_path, index=False)

    print(f"[✓] Cleaned dataset saved to: {output_path}")
    print(f"[✓] Total sentence pairs: {len(combined)}")

if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/en_hi.csv'